# Impetus — Energy-Based Reasoning for Open LLMs
## Kaggle Notebook | Baseline Benchmarking

**Strategy:** Math + Logic first (GSM8K, ARC, BBH)

**Pipeline:** Prompt → N candidates → Energy scoring → Best answer

---
### Instructions
1. Settings → **Accelerator → GPU T4 x1** (if P100 is assigned, notebook handles it)
2. **Internet** must be ON (Settings → Internet)
3. Run all cells
4. Results saved as CSV in `/kaggle/working/results/`
5. Download CSV or submit to GitHub

In [ ]:
import sys, os, subprocess
from pathlib import Path

# Detect GPU
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip().lower()
print(f"Detected GPU: {gpu_info}")

# P100 (sm_60) is incompatible with PyTorch >= 2.1; install older torch
if "p100" in gpu_info:
    print("P100 detected - installing PyTorch 2.0.1 (compatible with sm_60)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "torch==2.0.1", "torchvision==0.15.2",
        "--index-url", "https://download.pytorch.org/whl/cu117"])
else:
    print("Modern GPU detected - installing latest PyTorch")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "torch>=2.1.0", "torchvision>=0.16.0",
        "--index-url", "https://download.pytorch.org/whl/cu118"])

# Install rest of dependencies
DEPS = [
    "transformers>=4.36.0",
    "accelerate>=0.25.0",
    "datasets>=2.16.0",
    "evaluate>=0.4.1",
    "sentencepiece>=0.1.99",
    "scipy", "pandas", "tqdm",
]
for dep in DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", dep])

print("Dependencies installed.")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Device: {gpu_name}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # P100 doesn't support bfloat16; T4/V100/A100 do
    DTYPE = torch.float16 if "P100" in gpu_name else torch.bfloat16
    print(f"Using dtype: {DTYPE}")

In [ ]:
# Clone/pull project
REPO_URL = "https://github.com/EdhieBM/Impetus.git"
PROJECT_DIR = "/kaggle/working/Impetus"

if os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"])
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"Working in: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded successfully")

In [ ]:
# Phase 2: KONA verifier full benchmark on GSM8K
from benchmarks.run_reranker import run_reranker_gsm8k, save_results
from energy_module.scorers import MajorityVotingScorer
from benchmarks.run_reranker import batch_generate, extract_answer_number
from benchmarks.run_baseline import format_gsm8k_prompt, load_gsm8k

scorer = MajorityVotingScorer()
N_MAX = 20  # generate this many per question, then evaluate at N=5,10,20

# Load data
ds = load_gsm8k("test", 50)
questions = [ex["question"] for ex in ds]
ref_answers = [extract_answer_number(ex["answer"]) for ex in ds]

# Phase 1: Greedy baseline
print("Generating baselines (greedy)...")
baseline_prompts = [format_gsm8k_prompt(q) for q in questions]
baseline_outputs = batch_generate(model, tokenizer, baseline_prompts, do_sample=False, batch_size=8)
baseline_nums = [extract_answer_number(o) for o in baseline_outputs]
baseline_acc = sum(1 for i in range(50) if baseline_nums[i] == ref_answers[i]) / 50

# Phase 2: Generate pool of N_MAX candidates per question
print(f"Generating {N_MAX} candidates per question (50 x {N_MAX} = {50*N_MAX} total)...")
all_prompts = []
q_indices = []
for i, q in enumerate(questions):
    p = format_gsm8k_prompt(q)
    for _ in range(N_MAX):
        all_prompts.append(p)
        q_indices.append(i)

all_outputs = batch_generate(model, tokenizer, all_prompts, do_sample=True, temperature=0.7, batch_size=16)

# Group back by question
candidates_by_q = {i: [] for i in range(50)}
for idx, out in zip(q_indices, all_outputs):
    candidates_by_q[idx].append(out)

# Evaluate at different N values
print("\n" + "="*55)
print("GSM8K: Baseline vs Reranker at N=5, N=10, N=20")
print("="*55)
print(f"{'Method':<25} {'Correct':<10} {'Accuracy':<10}")
print("-"*45)
print(f"{'Baseline (greedy)':<25} {int(baseline_acc*50):<10} {baseline_acc:.2%}")

for N in [5, 10, 20]:
    correct = 0
    for i in range(50):
        pool = candidates_by_q[i][:N]
        scores = scorer.score_batch(questions[i], pool)
        scored = list(zip(pool, scores))
        best = min(scored, key=lambda x: x[1])[0]
        best_num = extract_answer_number(best)
        if best_num == ref_answers[i]:
            correct += 1
    acc = correct / 50
    delta = acc - baseline_acc
    print(f"{'Reranked N='+str(N):<25} {correct:<10} {acc:.2%} (Δ{delta:+.2%})")
print("="*55)

In [ ]:
print(f"\nModel: {MODEL_NAME} | GPU: {torch.cuda.get_device_name(0)}")
print("Results saved to /kaggle/working/Impetus/results/")